In [1]:
import json
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RAW_PATH = "/content/drive/MyDrive/Datasets/online_retail_transactions_raw.csv"
OUT_DIR = "/mnt/user-data/outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42

In [5]:
# ---------------------------------------------------------------------------
# 1. LOAD & CLEAN
# ---------------------------------------------------------------------------



print("Loading raw data...")
df = pd.read_csv(RAW_PATH)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print(f"Data shape: {df.shape}")
print(f"Data head: {df.head(5)}")

before = len(df)
df = df[df["CustomerID"].notna() & (df["CustomerID"] != 0)]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.upper().str.startswith("C")]  # cancellations
df["CustomerID"] = df["CustomerID"].astype(int)
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]
after = len(df)
print(f"Rows before cleaning: {before:,} | after cleaning: {after:,}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"Date range: {df['InvoiceDate'].min()} -> {df['InvoiceDate'].max()}")


Loading raw data...
Data shape: (531282, 8)
Data head:    InvoiceNo StockCode                          Description  Quantity  \
0     536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1     536365     71053                  WHITE METAL LANTERN         6   
2     536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3     536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4     536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55       17850  United Kingdom  
1 2010-12-01 08:26:00       3.39       17850  United Kingdom  
2 2010-12-01 08:26:00       2.75       17850  United Kingdom  
3 2010-12-01 08:26:00       3.39       17850  United Kingdom  
4 2010-12-01 08:26:00       3.39       17850  United Kingdom  
Rows before cleaning: 531,282 | after cleaning: 397,884
Unique customers: 4,338
Date range: 2010-12-01 08:26:00 -> 2011-12-09 12

In [6]:
# ---------------------------------------------------------------------------
# 2. CALIBRATION / HOLD-OUT SPLIT (avoids leakage in churn label)
# ---------------------------------------------------------------------------


max_date = df["InvoiceDate"].max()
min_date = df["InvoiceDate"].min()
HOLDOUT_DAYS = 90
cutoff = max_date - pd.Timedelta(days=HOLDOUT_DAYS)
print(f"Calibration period: {min_date.date()} -> {cutoff.date()}")
print(f"Hold-out period:    {cutoff.date()} -> {max_date.date()} "
      f"({HOLDOUT_DAYS} days, used only to define churn labels)")

calib = df[df["InvoiceDate"] <= cutoff].copy()
holdout = df[df["InvoiceDate"] > cutoff].copy()

# Snapshot date for Recency = end of calibration window (+1 day)
snapshot_date = cutoff + pd.Timedelta(days=1)

Calibration period: 2010-12-01 -> 2011-09-10
Hold-out period:    2011-09-10 -> 2011-12-09 (90 days, used only to define churn labels)


In [7]:
# ---------------------------------------------------------------------------
# 3. RFM + BEHAVIORAL FEATURES (calibration period only)
# ---------------------------------------------------------------------------
rfm = calib.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
    FirstPurchase=("InvoiceDate", "min"),
    UniqueProducts=("StockCode", "nunique"),
    TotalItems=("Quantity", "sum"),
).reset_index()

rfm["TenureDays"] = (snapshot_date - rfm["FirstPurchase"]).dt.days
rfm["AvgOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["AvgItemsPerOrder"] = rfm["TotalItems"] / rfm["Frequency"]

# Keep only customers who were actually active during calibration
rfm = rfm[rfm["Frequency"] > 0].reset_index(drop=True)
print(f"Customers with calibration-period activity: {len(rfm):,}")


Customers with calibration-period activity: 3,370


In [9]:
# ---------------------------------------------------------------------------
# 4. CHURN LABEL FROM HOLD-OUT PERIOD (no purchase in hold-out => churned)
# ---------------------------------------------------------------------------
returning_customers = set(holdout["CustomerID"].unique())
rfm["Churned"] = (~rfm["CustomerID"].isin(returning_customers)).astype(int)
churn_rate = rfm["Churned"].mean()
print(f"Observed churn rate in hold-out window: {churn_rate:.2%}")

Observed churn rate in hold-out window: 43.00%


In [10]:
# ---------------------------------------------------------------------------
# 5. K-MEANS SEGMENTATION ON RFM
# ---------------------------------------------------------------------------
rfm_log = rfm[["Recency", "Frequency", "Monetary"]].copy()
# log1p to tame heavy right skew typical of monetary/frequency retail data
rfm_log["Recency"] = np.log1p(rfm_log["Recency"])
rfm_log["Frequency"] = np.log1p(rfm_log["Frequency"])
rfm_log["Monetary"] = np.log1p(rfm_log["Monetary"].clip(lower=0))

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)

# Elbow + silhouette to choose k
from sklearn.metrics import silhouette_score

inertias, sil_scores, ks = [], [], list(range(2, 9))
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.plot(ks, inertias, marker="o", color="#2C6E49")
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("Inertia (elbow)", color="#2C6E49")
ax2 = ax1.twinx()
ax2.plot(ks, sil_scores, marker="s", color="#B5651D")
ax2.set_ylabel("Silhouette score", color="#B5651D")
plt.title("K-Means: Elbow & Silhouette Diagnostics")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_kmeans_elbow_silhouette.png"), dpi=150)
plt.close()

K_FINAL = 4
kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

# Profile clusters to assign human-readable segment names
profile = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean()
profile["CustomerCount"] = rfm.groupby("Cluster").size()
print("\nCluster centroid profile (raw units, mean):")
print(profile)


def name_segment(row, med_r, med_f, med_m):
    if row["Recency"] <= med_r and row["Frequency"] >= med_f and row["Monetary"] >= med_m:
        return "Champions / High-Value Loyal"
    if row["Recency"] <= med_r and row["Frequency"] < med_f:
        return "Promising / Recent New Customers"
    if row["Recency"] > med_r and row["Frequency"] >= med_f:
        return "At-Risk High-Value"
    return "Hibernating / Lost"


med_r, med_f, med_m = profile["Recency"].median(), profile["Frequency"].median(), profile["Monetary"].median()
profile["Segment"] = profile.apply(name_segment, axis=1, args=(med_r, med_f, med_m))
segment_map = profile["Segment"].to_dict()
rfm["Segment"] = rfm["Cluster"].map(segment_map)

print("\nSegment sizes:")
print(rfm["Segment"].value_counts())
print("\nChurn rate by segment:")
print(rfm.groupby("Segment")["Churned"].mean().sort_values(ascending=False))



Cluster centroid profile (raw units, mean):
            Recency  Frequency     Monetary  CustomerCount
Cluster                                                   
0        158.009859   1.203521   288.863092           1420
1         17.206250  12.077083  6781.835437            480
2         83.531476   3.402477  1523.361260            969
3         16.391218   2.201597   626.350341            501

Segment sizes:
Segment
Hibernating / Lost                  1420
At-Risk High-Value                   969
Promising / Recent New Customers     501
Champions / High-Value Loyal         480
Name: count, dtype: int64

Churn rate by segment:
Segment
Hibernating / Lost                  0.628169
Promising / Recent New Customers    0.409182
At-Risk High-Value                  0.329205
Champions / High-Value Loyal        0.068750
Name: Churned, dtype: float64


In [15]:
# --- Visualization: RFM distributions ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    sns.histplot(rfm[col], bins=40, ax=ax, color="#3E7CB1")
    ax.set_title(f"Distribution of {col}")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "02_rfm_distributions.png"), dpi=150)
plt.show()
plt.close()

# --- Visualization: cluster scatter (Frequency vs Monetary, colored by segment) ---
plt.figure(figsize=(7.5, 5.5))
palette = sns.color_palette("Set2", n_colors=rfm["Segment"].nunique())
sns.scatterplot(
    data=rfm, x="Recency", y="Monetary", hue="Segment", palette=palette, alpha=0.6, s=25
)
plt.yscale("log")
plt.title("Customer Segments: Recency vs Monetary Value (log scale)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "03_segment_scatter.png"), dpi=150)
plt.show()
plt.close()

In [16]:
# ---------------------------------------------------------------------------
# 6. CLASSIFICATION MODELS: PREDICT HOLD-OUT CHURN
# ---------------------------------------------------------------------------
feature_cols = [
    "Recency", "Frequency", "Monetary", "TenureDays",
    "UniqueProducts", "AvgOrderValue", "AvgItemsPerOrder",
]
X = rfm[feature_cols].copy()
y = rfm["Churned"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler_clf = StandardScaler()
X_train_scaled = scaler_clf.fit_transform(X_train)
X_test_scaled = scaler_clf.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_depth=8, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE
    ),
}

metrics = {}
plt.figure(figsize=(6.5, 5.5))
ax = plt.gca()
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    metrics[name] = {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
        "f1": round(f1_score(y_test, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_test, y_proba), 4),
    }
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

    RocCurveDisplay.from_predictions(y_test, y_proba, name=name, ax=ax)

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("ROC Curves: Churn Classification Models")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "04_roc_curves.png"), dpi=150)
plt.close()



=== Logistic Regression ===
              precision    recall  f1-score   support

    Retained       0.77      0.58      0.66       481
     Churned       0.58      0.77      0.66       362

    accuracy                           0.66       843
   macro avg       0.68      0.68      0.66       843
weighted avg       0.69      0.66      0.66       843

Confusion matrix:
 [[281 200]
 [ 84 278]]

=== Random Forest ===
              precision    recall  f1-score   support

    Retained       0.73      0.58      0.65       481
     Churned       0.56      0.72      0.63       362

    accuracy                           0.64       843
   macro avg       0.65      0.65      0.64       843
weighted avg       0.66      0.64      0.64       843

Confusion matrix:
 [[280 201]
 [103 259]]


In [17]:
# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (name, model) in zip(axes, models.items()):
    Xin = X_test_scaled if name == "Logistic Regression" else X_test
    y_pred = model.predict(Xin)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Retained", "Churned"], yticklabels=["Retained", "Churned"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "05_confusion_matrices.png"), dpi=150)
plt.close()

# Random Forest feature importance
rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values()
plt.figure(figsize=(7, 4.5))
importances.plot(kind="barh", color="#4C6E5D")
plt.title("Random Forest Feature Importance (Churn Prediction)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "06_feature_importance.png"), dpi=150)
plt.close()

print("\nModel comparison:")
print(json.dumps(metrics, indent=2))

with open(os.path.join(OUT_DIR, "model_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)



Model comparison:
{
  "Logistic Regression": {
    "accuracy": 0.6631,
    "precision": 0.5816,
    "recall": 0.768,
    "f1": 0.6619,
    "roc_auc": 0.7263
  },
  "Random Forest": {
    "accuracy": 0.6394,
    "precision": 0.563,
    "recall": 0.7155,
    "f1": 0.6302,
    "roc_auc": 0.7153
  }
}


In [18]:
# ---------------------------------------------------------------------------
# 7. SCORE ALL CUSTOMERS WITH BEST MODEL & BUILD TARGETING MATRIX
# ---------------------------------------------------------------------------
best_model_name = max(metrics, key=lambda m: metrics[m]["roc_auc"])
print(f"\nBest model by ROC-AUC: {best_model_name}")
best_model = models[best_model_name]

if best_model_name == "Logistic Regression":
    X_all_scaled = scaler_clf.transform(X)
    rfm["ChurnProbability"] = best_model.predict_proba(X_all_scaled)[:, 1]
else:
    rfm["ChurnProbability"] = best_model.predict_proba(X)[:, 1]

rfm["RiskTier"] = pd.cut(
    rfm["ChurnProbability"], bins=[-0.01, 0.33, 0.66, 1.0],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

# Prioritized targeting matrix: Segment x Risk Tier -> customer count & revenue at stake
targeting_matrix = rfm.pivot_table(
    index="Segment", columns="RiskTier", values="Monetary",
    aggfunc=["count", "sum"], observed=False
).fillna(0)
targeting_matrix.to_csv(os.path.join(OUT_DIR, "targeting_matrix_segment_x_risk.csv"))
print("\nPrioritized targeting matrix (customer count / revenue at stake):")
print(targeting_matrix)

# Final master customer table
final_cols = [
    "CustomerID", "Recency", "Frequency", "Monetary", "TenureDays",
    "UniqueProducts", "AvgOrderValue", "AvgItemsPerOrder",
    "Cluster", "Segment", "Churned", "ChurnProbability", "RiskTier",
]
rfm[final_cols].to_csv(os.path.join(OUT_DIR, "customer_rfm_segments_churn.csv"), index=False)

print("\nDone. Outputs written to:", OUT_DIR)



Best model by ROC-AUC: Logistic Regression

Prioritized targeting matrix (customer count / revenue at stake):
                                    count                               sum  \
RiskTier                         Low Risk Medium Risk High Risk    Low Risk   
Segment                                                                       
At-Risk High-Value                    298         635        36   551252.26   
Champions / High-Value Loyal          478           2         0  3249876.97   
Hibernating / Lost                      0         529       891        0.00   
Promising / Recent New Customers       65         435         1    67312.75   

                                                         
RiskTier                         Medium Risk  High Risk  
Segment                                                  
At-Risk High-Value                785677.671  139207.13  
Champions / High-Value Loyal        5404.040       0.00  
Hibernating / Lost                162657.720 